# Model Training and Evaluation

## Objective

This notebook compares machine-learning models for binary cardiovascular disease classification using the UCI Heart Disease dataset.

## Research Questions

1. How does Logistic Regression compare with tree-based models?
2. Do different models show consistent performance under stratified cross-validation?
3. What trade-offs exist between precision, recall, F1-score and ROC-AUC?

## Experimental Strategy

- Use the existing training and test split.
- Keep preprocessing inside each model pipeline to prevent data leakage.
- Compare models using the same stratified cross-validation folds.
- Select models using training data only.
- Reserve the held-out test set for final evaluation after model selection.

## Scope and Limitations

The dataset contains 303 observations. Results will be interpreted cautiously because performance estimates from a small dataset can be uncertain. This project is research-oriented and is not intended for clinical diagnosis or medical decision-making.


In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split

# Loading data
df = pd.read_csv("../data/raw/heart_disease_data.csv")

# Separating predictors and target
X = df.drop(columns=["num"]).copy()
y = (df["num"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

Training set: (242, 13)
Test set: (61, 13)

Training target distribution:
num
0    131
1    111
Name: count, dtype: int64

Test target distribution:
num
0    33
1    28
Name: count, dtype: int64


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_features = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

# Numerical preprocessing
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combining both pipelines
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Logistic Regression pipeline
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Defining 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# Evaluating on training data only
logistic_results = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

# results
for metric in scoring:
    scores = logistic_results[f"test_{metric}"]
    print(
        f"{metric.upper():10s}: "
        f"{scores.mean():.3f} +/- {scores.std():.3f}"
    )

ACCURACY  : 0.847 +/- 0.010
PRECISION : 0.877 +/- 0.054
RECALL    : 0.783 +/- 0.046
F1        : 0.825 +/- 0.007
ROC_AUC   : 0.902 +/- 0.014


In [17]:
print("Dataset shape:", df.shape)

print("\nTraining data shape:", X_train.shape)

print("\nTarget distribution:")
print(y_train.value_counts())

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nCross-validation:")
print(cv)

Dataset shape: (303, 14)

Training data shape: (242, 13)

Target distribution:
num
0    131
1    111
Name: count, dtype: int64

Numerical features:
['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

Categorical features:
['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

Cross-validation:
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [18]:
# Feature definitions
num_features = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

cat_features = [
    "sex", "cp", "fbs", "restecg",
    "exang", "slope", "ca", "thal"
]

# Fresh numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Fresh categorical preprocessing
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Fresh combined preprocessing
fresh_preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

# Fresh Logistic Regression model
fresh_model = Pipeline([
    ("preprocessor", fresh_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Fresh cross-validation
fresh_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation
fresh_results = cross_validate(
    fresh_model,
    X_train,
    y_train,
    cv=fresh_cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    error_score="raise"
)

# Printing individual fold scores
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    scores = fresh_results[f"test_{metric}"]

    print(f"\n{metric.upper()}")
    print("Individual folds:", scores)
    print(f"Mean: {scores.mean():.4f}")
    print(f"Std:  {scores.std():.4f}")


ACCURACY
Individual folds: [0.83673469 0.85714286 0.85416667 0.83333333 0.85416667]
Mean: 0.8471
Std:  0.0100

PRECISION
Individual folds: [0.82608696 0.94117647 0.94117647 0.81818182 0.85714286]
Mean: 0.8768
Std:  0.0542

RECALL
Individual folds: [0.82608696 0.72727273 0.72727273 0.81818182 0.81818182]
Mean: 0.7834
Std:  0.0459

F1
Individual folds: [0.82608696 0.82051282 0.82051282 0.81818182 0.8372093 ]
Mean: 0.8245
Std:  0.0069

ROC_AUC
Individual folds: [0.9180602  0.90909091 0.88986014 0.91433566 0.88111888]
Mean: 0.9025
Std:  0.0144


In [21]:
# applying Random Forest

from sklearn.ensemble import RandomForestClassifier

# Random Forest pipeline
rf_model = Pipeline(steps=[
    ("preprocessor", fresh_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

# 5-fold cross-validation
rf_results = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=fresh_cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }
)

# Displaying results
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    scores = rf_results[f"test_{metric}"]

    print(
        f"{metric.upper():10s}: "
        f"{scores.mean():.3f} +/- {scores.std():.3f}"
    )

ACCURACY  : 0.814 +/- 0.023
PRECISION : 0.797 +/- 0.027
RECALL    : 0.801 +/- 0.080
F1        : 0.796 +/- 0.036
ROC_AUC   : 0.894 +/- 0.033


### Initial Model Comparison: Logistic Regression vs Random Forest

Both models were evaluated using 5-fold stratified cross-validation on the same training dataset, with identical preprocessing and evaluation metrics.

Logistic Regression achieved better accuracy (0.847 ± 0.010), precision (0.877 ± 0.054), F1-score (0.825 ± 0.007), and ROC-AUC (0.902 ± 0.014) compared with Random Forest.

Random Forest achieved marginally higher recall (0.801 ± 0.080) than Logistic Regression (0.783 ± 0.046).

Logistic Regression also showed lower variability across all five evaluation metrics.

**Preliminary observation:** Logistic Regression currently provides the more consistent overall performance. However, further evaluation is required before selecting the final model.